# LangChain Memory — 2026
## Notebook 0: The Big Picture + Setup

Welcome! Yesterday we learned **prompts** (PromptTemplate, ChatPromptTemplate, MessagesPlaceholder).
Today: **memory** — how an LLM app *remembers* things.

### The one-sentence version
An LLM by itself has **zero memory**. Every call starts blank.
"Memory" in LangChain = **saving the conversation and giving it back to the LLM on the next call.**

### Think of it like a human
| Human | LangChain |
|---|---|
| What you remember **during one phone call** | **Short-term memory** (one conversation / "thread") |
| The **notebook** where you write things down to remember next week | **Long-term memory** (shared across all conversations) |

### Today's map
```
LANGCHAIN MEMORY
├── SHORT-TERM (current thread)
│     └── Agent State  → saved by a Checkpointer
│           ├── full history ...... Notebook 1
│           ├── window (trim) ..... Notebook 2
│           └── summary ........... Notebook 3
└── LONG-TERM (across threads)
      └── Store
            ├── Semantic ... facts & preferences ... Notebook 4
            ├── Episodic ... past experiences ...... Notebook 5
            └── Procedural . rules & behavior ...... Notebook 6
```

### Old vs New (so you don't get confused by old tutorials)
| Old LangChain (legacy, deprecated) | Modern LangChain (2026) |
|---|---|
| `ConversationBufferMemory` | `MessagesState` + **Checkpointer** |
| `ConversationBufferWindowMemory` | `trim_messages()` before the LLM call |
| `ConversationSummaryMemory` | `SummarizationNode` (from `langmem`) |

Same ideas, new home: memory now lives in the **agent's state**, not in a separate memory object.

## Setup

1. Install the packages (run the next cell once).
2. Create a file named **`.env`** in this folder with your key:

```
OPENAI_API_KEY=sk-your-key-here
```

3. Every notebook starts with `load_dotenv()` — it reads that file so the key is never written in code.

In [ ]:
# Run this cell ONCE (remove the # first if packages are missing)
# %pip install langchain langchain-openai langgraph langmem python-dotenv
print("Skip me if packages are already installed.")

In [ ]:
# Every notebook begins exactly like this:
from dotenv import load_dotenv
load_dotenv()   # reads OPENAI_API_KEY from the .env file

import os
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found — check your .env file!"
print("API key loaded successfully.")

In [ ]:
# Quick test: one stateless call to the LLM (no memory yet!)
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Watch: two separate calls. The LLM forgets the first one immediately.
llm.invoke("My name is Rahul.")
print(llm.invoke("What is my name?").content)
# -> It does NOT know. That is the problem memory solves. On to Notebook 1!

### How to use these notebooks
Read each markdown cell, then run the code cell below it and **look at the output** before moving on.
Every notebook ends with a **"Try it yourself"** — change the code and predict what happens first.

**How the notebooks connect:** Notebook 1 is the foundation (state + checkpointer). Notebooks 2 & 3 only change *what the LLM sees* from that state. Notebooks 4–6 add a second component — the **Store** — to remember things *across* conversations.